# Question-Answering Chatbot


Objective:
- To create a context-aware question answering chatbot by retrieving and sythesizing information from company policy document or pdf uploaded document.

Business Use Cases
- Employee self-service support: Allow employees to instantly query company policy documents or uploaded PDF without relying on HR personnel or manual document searches.

ML Framing:
- Type: Information Retrieval + Generative NLP (Retrieval-Augmented Generation)
- Input: User query + retrieved policy text chunks
- Output: Natural language answer grounded from the documents and source citations such as page number
- Constraints: Answer only from retrieved documents; refuse if information is unavailable

Note: 
- The chatbot is designed to answer only from retrieved policy content or uploaded PDF.
- It will explicitly refuse to answer when the information is not present in the documents

### Architecture Overview

1. PDF ingestion and chunking
2. Semantic embedding using OpenAI embeddings
3. Vector storage using ChromaDB with persistence
4. Retrieval of top-k relevant chunks
5. Answer generation using an LLM constrained by retrieved context


### Preprocessing

1. Import libraries
2. Load OpenAI API key
3. Set constant
4. Set path for chroma_store
5. Set path for Company Policy PDF

1) Import libraries

We’ll persist vector stores so indexing happens only once per document.


In [1]:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

from __future__ import annotations

from dotenv import load_dotenv
import os
import hashlib
from pathlib import Path
from typing import List, Optional

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

import os
os.chdir(r'C:\Users\ashle\Project\usecase')

from src.functions.chatbot_pipeline import build_demo

2. Load OpenAI API key in .env file and load .env

In [2]:
load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "Missing OPENAI_API_KEY env var. Set it and restart kernel."

3. Set constants

In [3]:
CHUNK_SIZE = 900
CHUNK_OVERLAP = 150
TOP_K = 4

LLM_MODEL = "gpt-4o-mini"
EMB_MODEL = "text-embedding-3-small"

4. Set path for chroma_store
    - chroma_store is a local generated vector index cache where Chroma (vector database) saves the index of the PDF, so it does not have to re-process the same PDF every time
    - chroma_store contains:
        - Embeddings for each text chunk (numeric vectos)
        - The vector index used for similarity search, enabled fast relevant chunks retrieval
        - Metada for each chunk, such as source, page, chunk ids
    - Normal run (time costly): load PDF -> split into chunks -> embed chunks -> build Chroma
    - By having chroma_store, the next run can reuse the saved vector DB, by retrieving the top-k chunks quickly

In [4]:
PERSIST_BASE = Path("chroma_store")
PERSIST_BASE.mkdir(exist_ok=True)

In [5]:
def persist_dir_for(doc_id: str):
    return PERSIST_BASE / doc_id[:16]

5. Set path for Company Policy PDF

In [6]:
PDF_PATH = Path(r'data\2025-Vistra-Code-of-Conduct.pdf')
assert PDF_PATH.exists(), f'PDF not found: {PDF_PATH}'

### Vector Store Setup

 1. Hashing
 2. Loading
 3. Chunking
 4. Embedding
 5. Build or load the vector store

1. Hashing
- Hash the PDF to create a stable cache key
- Uploading the same PDF again becomes instant

In [7]:
def sha256_file(path: str | Path, block_size: int = 1 << 20):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            chunk = f.read(block_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

2. Loading

In [8]:
def load_pdf(path: str | Path):
    path = Path(path)
    loader = PyPDFLoader(str(path))
    docs = loader.load()
    # Ensure consistent metadata fields for citation
    for d in docs:
        d.metadata.setdefault("source", str(path.name))
    return docs

3. Chunking

In [9]:
def split_docs(docs: List[Document], chunk_size: int = CHUNK_SIZE, chunk_overlap: int = CHUNK_OVERLAP):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    return splitter.split_documents(docs)

4. Embedding

In [10]:
def get_embeddings():
    return OpenAIEmbeddings(model=EMB_MODEL)

5. Build or load the vector store

In [11]:
def load_or_build_vectordb(pdf_path: str | Path, *, doc_id: Optional[str] = None):
    pdf_path = Path(pdf_path)
    if doc_id is None:
        doc_id = sha256_file(pdf_path)

    pdir = persist_dir_for(doc_id)
    embeddings = get_embeddings()

    # If directory exists and looks non-empty, load it
    if pdir.exists() and any(pdir.iterdir()):
        vectordb = Chroma(persist_directory=str(pdir), embedding_function=embeddings)
        return doc_id, vectordb

    # Otherwise build it
    pdir.mkdir(parents=True, exist_ok=True)
    docs = load_pdf(pdf_path)
    chunks = split_docs(docs)
    
    # Create the store
    vectordb = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=str(pdir),
    )
    vectordb.persist()
    return doc_id, vectordb

### Modeling

1. Set system rules
2. Set LLM model
3. Set context metadata
4. Question-Answer model
5. Inspect top-k chunks retrieval
6. Test the question-answer LLM model

1. Set system rules

In [12]:
SYSTEM_RULES = """
You are a company policy assistant.

Rules:
1) Answer ONLY using the provided policy context.
2) If the answer is not in the context, say: "I can't find that in the provided policy documents."
3) When you answer, include citations in the format [source p.X] for each key claim.
4) Keep answers concise and practical.
""".strip()


PROMPT = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_RULES),
    ("human", "Question: {question}\n\nPolicy context:\n{context}"),
])

2. Set LLM model

In [13]:
def get_llm():
    return ChatOpenAI(model=LLM_MODEL, temperature=0.2)

3. Set context metadata
- Include page metadata for the model to cite

In [14]:
def format_context(docs: List[Document]):
    # Include page metadata for the model to cite
    lines = []
    for i, d in enumerate(docs, start=1):
        src = d.metadata.get("source", "unknown")
        page = d.metadata.get("page", None)
        page_str = f"p.{page+1}" if isinstance(page, int) else "p.?"
        lines.append(f"--- Snippet {i} ({src} {page_str}) ---\n{d.page_content}")
    return "\n\n".join(lines)

4. Question-Answer model
- Retrieve top-k chunks
- LLM to answer only from context
- If context does not match, it should refuse

In [15]:
def answer_question(vectordb: Chroma, question: str, *, k: int = TOP_K):
    retriever = vectordb.as_retriever(search_kwargs={"k": k})
    docs = retriever.invoke(question)
    

    llm = get_llm()
    context = format_context(docs)
    
    msg = PROMPT.format_messages(question=question, context=context)
    resp = llm.invoke(msg)


    return {
        "answer": resp.content,
        "sources": docs,
    }

In [16]:
doc_id, vectordb = load_or_build_vectordb(PDF_PATH)
print("Company policy:", PDF_PATH.name, "doc_id:", doc_id[:12])

Company policy: 2025-Vistra-Code-of-Conduct.pdf doc_id: a64a3aa5ac56


5. Inspect top-k chunks retrieval

- To confirm the retriever returns relevant snippets and page metadata before generating the final answer

In [17]:
# Inspect retrieved chunks BEFORE generating an answer
question = "What is the smoking policy?"
retriever = vectordb.as_retriever(search_kwargs={"k": TOP_K})
docs = retriever.invoke(question)

print("Retrieved chunks:", len(docs))
print()
print(format_context(docs)[:2000])  # preview first ~2000 chars

Retrieved chunks: 4

--- Snippet 1 (C:\Users\ashle\Project\usecase\data\2025-Vistra-Code-of-Conduct.pdf p.14) ---
privately owned motor vehicle that is parked 
in an area the Company has designated for 
employee parking. 
Q A
Employee Assistance Program
The Company’s Employee Assistance Program is 
designed to confidentially help employees and 
their dependents manage issues with emotional, 
marital, family, or other personal difficulties, 
including dependency on drugs or alcohol. 
Smoking
Smoking is prohibited in all Company buildings, 
facilities, vehicles, and equipment owned or 
leased by the Company unless otherwise 
provided in the Workplace Conduct Policy. This 
includes vaping devices and e-cigarettes. 
Possession of Weapons  
and Firearms
Company policy prohibits the possession of 
weapons, firearms (with or without a license), 
and ammunition whether classified as legal or 
illegal on Company property, including buildings, 
parking lots, recreation facilities, equipment, and

6. Test the question-answer LLM model

In [18]:
question = 'What is the smoking policy?'

out = answer_question(vectordb, question)
print(out["answer"])

The smoking policy prohibits smoking in all Company buildings, facilities, vehicles, and equipment owned or leased by the Company. This prohibition includes vaping devices and e-cigarettes, unless otherwise specified in the Workplace Conduct Policy [source p.14].


In [19]:
unanswerable = 'What is the company policy on Mars travel reimbursement?'
out2 = answer_question(vectordb, unanswerable)
print(out2['answer'])

I can't find that in the provided policy documents.


In [20]:
question = 'Provide summary of this document?'

out = answer_question(vectordb, question)
print(out["answer"])

The document is the Vistra Code of Conduct, which outlines the company's policies and expectations regarding ethical behavior and compliance in various areas. Key sections include:

1. **Workplace Conduct**: Emphasizes fairness, inclusion, health and safety, and respect, addressing issues like harassment and workplace violence [source p.2].
2. **Use of Company Assets**: Covers intellectual property, confidentiality, data privacy, and the appropriate use of company resources [source p.2].
3. **Conflicts of Interest**: Discusses the importance of avoiding conflicts through guidelines on gifts, entertainment, and outside activities [source p.2].
4. **Relationships with Stakeholders**: Addresses interactions with customers, suppliers, and competitors, including antitrust laws and procurement activities [source p.2].
5. **Compliance and Reporting**: Provides a compliance helpline and encourages reporting of unethical behavior [source p.19].

The document serves as a comprehensive guide to m

### Question-Answering Chatbot Demo

Example Questions:
- How should employees report workplace misconduct?
- Is personal device usage allowed on company networks?


In [21]:
demo = build_demo()

demo.launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.
